# GNN-ReservoirNet — Kaggle training run

Trains the STGNN on the verified `wris_v2` dataset (10 Peninsular-India reservoirs,
2010–2024, real gauge/at-dam inflow + cleaned storage, IOD-aware climate features).

**Attach ONE of these as an Input before running:**
1. GitHub repo `rajhodedara/gnn-reservoirnet` (Kaggle Settings → GitHub → connect, then Add Input), or
2. a Kaggle Dataset uploaded from the local `kaggle_stage/` folder.

**Output artifacts** (saved under `/kaggle/working/gnn-reservoirnet/runs/`):
`best_model_finetune.pt`, evaluation metric CSVs, explainability report.

**Baseline bar to beat** (test 2024, pooled NSE): persistence **0.478** / climatology **0.495**.

Note: without ERA5 NetCDFs attached, rainfall/evap/soil-moisture features are zero-filled
and the model trains on inflow + storage + climate indices — fine for run #1.

In [ ]:
# 1. Get the code + data into the writable working dir
#    (public repo -> plain git clone works; attached inputs are used if present)
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/rajhodedara/gnn-reservoirnet.git"
dst = Path("/kaggle/working/gnn-reservoirnet")

src = next((c for c in [
    "/kaggle/input/gnn-reservoirnet",
    "/kaggle/input/gnn-reservoirnet-dataset",
] if Path(c).exists()), None)

if dst.exists():
    shutil.rmtree(dst)

if src:
    shutil.copytree(src, dst, ignore=shutil.ignore_patterns("__pycache__", ".ipynb_checkpoints"))
    print("Copied attached input:", src)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dst)],
                       capture_output=True, text=True)
    assert r.returncode == 0, f"git clone failed: {r.stderr}"
    print("Cloned public repo:", REPO_URL)

os.chdir(dst)
print("Working in:", os.getcwd())

In [ ]:
# 2. Install only what Kaggle lacks (torch is preinstalled)
!pip install -q torch-geometric properscoring

In [ ]:
# 3. Sanity checks: dataset complete + IOD present
import glob
import pandas as pd

files = sorted(glob.glob("data/raw/wris_v2/*.csv"))
assert len(files) == 10, f"Expected 10 reservoir CSVs, found {len(files)}: {files}"
cli = pd.read_csv("data/raw/enso/combined_climate_indices.csv")
assert "iod" in cli.columns, "IOD missing — the climate CSV is stale; re-merge with scripts/merge_iod.py"
for f in files:
    n = sum(1 for _ in open(f, encoding="utf-8")) - 1
    assert n == 5479, f"{f}: {n} rows != 5479"
print("Data OK:", len(files), "reservoirs x 5479 days; climate columns:", list(cli.columns))

In [ ]:
# 4. Smoke-check the test suite on the Kaggle environment (optional but recommended)
!python -m pytest tests/ -q --no-header

In [ ]:
# 5. TRAIN — 200 epochs, early stopping (patience 20), GPU recommended (T4/P100)
#    Trains on 2010-2022, validates on 2023, then evaluates the BEST checkpoint on 2024.
!python main.py --config configs/default_config.yaml

In [ ]:
# 6. Recompute the baselines in the same environment for a fair comparison
!python scripts/run_baselines.py --config configs/default_config.yaml

In [ ]:
# 7. Artifact inventory — /kaggle/working is saved as the notebook output
from pathlib import Path

for p in sorted(Path("runs").rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size:>10,}  {p}")
print("\nDownload runs/ + outputs/ from the notebook Output tab, then hand them back for verification.")